# EEG Dataset Unit Check & Loader Test
KL 2026-01-06

In [101]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
from loader import load_dataset, split_dataset, SplitStrategy
from hdf5_io import HDF5Reader
import numpy as np

In [102]:
def detect_unit(data: np.ndarray) -> tuple[str, float]:
    """Detect data unit based on amplitude range."""
    max_amp = np.abs(data).max()
    if max_amp > 1e1:  # > 10, likely µV
        return "µV", max_amp
    return "V", max_amp


def check_dataset_unit(data_dir: str) -> dict:
    """Check unit and amplitude statistics for a dataset."""
    h5_files = sorted(Path(data_dir).glob("sub_*.h5"))
    if not h5_files:
        return {"error": "No HDF5 files found"}

    stats = {
        "dataset_name": None,
        "num_files": len(h5_files),
        "num_labels": None,
        "category_list": None,
        "detected_unit": None,
        "max_amplitude": 0,
        "min_amplitude": float('inf'),
        "mean_amplitude": [],
        "samples_checked": 0,
        "amplitude_violations": 0,
    }

    for h5_file in h5_files[:3]:
        print(f"Checking file: {h5_file}")
        with HDF5Reader(str(h5_file)) as reader:
            subj = reader.subject_attrs
            print(subj)
            if stats["dataset_name"] is None:
                stats["dataset_name"] = subj.dataset_name
                stats["num_labels"] = getattr(subj, 'num_labels', None)
                stats["category_list"] = getattr(subj, 'category_list', None)

            trial_names = reader.get_trial_names()[:3]
            for trial_name in trial_names:
                seg_names = reader.get_segment_names(trial_name)[:5]
                # `subj = reader.subject_attrs` is assigning the attribute `subject_attrs` of the `reader`
                # object to the variable `subj`. This line of code is retrieving the subject attributes from
                # the HDF5 file being read by the `reader` object. These attributes typically contain
                # information about the dataset, task type, subject ID, sampling rate, montage, channels, and
                # other relevant metadata related to the EEG data being processed.
                for seg_name in seg_names:
                    seg = reader.get_segment(trial_name, seg_name)
                    data = seg.data
                    
                    max_amp = np.abs(data).max()
                    
                    stats["max_amplitude"] = max(stats["max_amplitude"], max_amp)
                    stats["min_amplitude"] = min(stats["min_amplitude"], max_amp)
                    stats["mean_amplitude"].append(np.abs(data).mean())
                    stats["samples_checked"] += 1
                    print(f"  Segment: {seg_name}, Max Amplitude: {max_amp:.2f}, mean Amplitude: {np.abs(data).mean():.2f}, median Amplitude: {np.median(np.abs(data)):.2f}")
                    if max_amp > 600:
                        stats["amplitude_violations"] += 1

    stats["detected_unit"], _ = detect_unit(np.array([stats["max_amplitude"]]))
    stats["mean_amplitude"] = np.mean(stats["mean_amplitude"]) if stats["mean_amplitude"] else 0
    return stats

## Single Dataset Report

In [103]:
# Change this to your dataset path
data_dir = "/mnt/dataset2/zlj/TUAE4-6"
# data_dir = "/mnt/dataset2/Processed_datasets/BCIC2A/BCIC2A"
# data_dir = "/mnt/dataset2/Processed_datasets/BCIC2A"
# data_dir = '/mnt/dataset2/hdf5_datasets/TUEV'
# data_dir = '/mnt/dataset2/Processed_datasets/EEG_Pre/TUH'

In [104]:
stats = check_dataset_unit(data_dir)
print(f"Dataset: {stats['dataset_name']}")
print(f"Files: {stats['num_files']}")
print(f"Samples checked: {stats['samples_checked']}")
print()
print("--- Label Info ---")
print(f"Num labels: {stats['num_labels']}")
print(f"Categories: {stats['category_list']}")
print()
print("--- Unit Analysis ---")
print(f"Detected unit: {stats['detected_unit']}")
print(f"Max amplitude: {stats['max_amplitude']:.4e}")
print(f"Min amplitude: {stats['min_amplitude']:.4e}")
print(f"Mean amplitude: {stats['mean_amplitude']:.4e}")
print()
print("--- Validation ---")
is_uv = stats['detected_unit'] == 'µV'
has_labels = stats['num_labels'] is not None and stats['num_labels'] > 0
has_categories = stats['category_list'] is not None and len(stats['category_list']) > 0
print(f"Unit is µV: {'✓' if is_uv else '✗ (NEEDS UPDATE)'}")
print(f"Has num_labels: {'✓' if has_labels else '✗ (NEEDS UPDATE)'}")
print(f"Has category_list: {'✓' if has_categories else '✗ (NEEDS UPDATE)'}")
if is_uv:
    print(f"Amplitude violations (>600 µV): {stats['amplitude_violations']}/{stats['samples_checked']}")

Checking file: /mnt/dataset2/zlj/TUAE4-6/sub_1.h5
SubjectAttrs(subject_id=1, dataset_name='TUAE_EEG_Dataset', task_type='resting_state', downstream_task_type='CLASSIFICATION', rsFreq=250.0, chn_name=['FP1', 'FP2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'A1', 'A2', 'FZ', 'CZ', 'PZ', 'T1', 'T2'], num_labels=2, category_list=['normal', 'abnormal'], chn_pos=None, chn_ori=None, chn_type='EEG', montage='10_20')
  Segment: segment0, Max Amplitude: 0.04, mean Amplitude: 0.01, median Amplitude: 0.01
  Segment: segment1, Max Amplitude: 0.05, mean Amplitude: 0.01, median Amplitude: 0.01
  Segment: segment10, Max Amplitude: 0.08, mean Amplitude: 0.01, median Amplitude: 0.01
  Segment: segment100, Max Amplitude: 0.05, mean Amplitude: 0.01, median Amplitude: 0.01
  Segment: segment101, Max Amplitude: 0.03, mean Amplitude: 0.01, median Amplitude: 0.01
Checking file: /mnt/dataset2/zlj/TUAE4-6/sub_10.h5
SubjectAttrs(subject_id=10, dataset_name='TUAE_EEG_Data

## testing hbn dataset

In [ ]:
import os
os.listdir('/mnt/dataset2/Datasets/HBN-EEG/ds005505-download/sub-NDARAC904DMU/eeg/')

['sub-NDARAC904DMU_task-DespicableMe_channels.tsv',
 'sub-NDARAC904DMU_task-DespicableMe_eeg.json',
 'sub-NDARAC904DMU_task-DiaryOfAWimpyKid_channels.tsv',
 'sub-NDARAC904DMU_task-DespicableMe_events.tsv',
 'sub-NDARAC904DMU_task-DiaryOfAWimpyKid_eeg.json',
 'sub-NDARAC904DMU_task-FunwithFractals_channels.tsv',
 'sub-NDARAC904DMU_task-FunwithFractals_eeg.json',
 'sub-NDARAC904DMU_task-DiaryOfAWimpyKid_events.tsv',
 'sub-NDARAC904DMU_task-RestingState_channels.tsv',
 'sub-NDARAC904DMU_task-FunwithFractals_events.tsv',
 'sub-NDARAC904DMU_task-DespicableMe_eeg.set',
 'sub-NDARAC904DMU_task-DiaryOfAWimpyKid_eeg.set',
 'sub-NDARAC904DMU_task-FunwithFractals_eeg.set',
 'sub-NDARAC904DMU_task-RestingState_eeg.json',
 'sub-NDARAC904DMU_task-RestingState_events.tsv',
 'sub-NDARAC904DMU_task-ThePresent_channels.tsv',
 'sub-NDARAC904DMU_task-ThePresent_eeg.json',
 'sub-NDARAC904DMU_task-ThePresent_events.tsv',
 'sub-NDARAC904DMU_task-RestingState_eeg.set',
 'sub-NDARAC904DMU_task-contrastChangeDe

In [ ]:
import mne
path = '/mnt/dataset2/Datasets/HBN-EEG/ds005505-download/sub-NDARAC904DMU/eeg//sub-NDARAC904DMU_task-contrastChangeDetection_run-1_eeg.set'

In [ ]:
test = mne.io.read_raw_eeglab(path)

/tmp/ipykernel_9091/2792990184.py:1: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  test = mne.io.read_raw_eeglab(path)


In [ ]:
test._data.shape

(129, 117052)

## Multi-Dataset Report

In [108]:
# Base directory containing all datasets
base_dir = "/mnt/dataset2/zlj"

In [109]:
base_path = Path(base_dir)
datasets = sorted([d for d in base_path.iterdir() if d.is_dir()])
print(datasets)
results = []


for dataset_dir in datasets:
    print(f"当前遍历目录：{dataset_dir}")
    h5_files = list(dataset_dir.glob("**/sub_*.h5"))
    print(f"该目录下找到的H5文件：{h5_files}")
    if not h5_files:
        print("未找到符合条件的H5文件，跳过")
        continue
    stats = check_dataset_unit(str(dataset_dir))
    stats["path"] = dataset_dir.name
    results.append(stats)
    print(f"已处理该目录，results当前长度：{len(results)}")

# Print summary table
print(f"{'Dataset':<20} {'Unit':<6} {'Max Amp':<12} {'Labels':<8} {'Categories':<10}")
print("-" * 70)

for r in results:
    if "error" in r:
        print(f"{r.get('path', 'Unknown'):<20} ERROR")
        continue
    unit_ok = "✓" if r['detected_unit'] == 'µV' else "✗"
    labels_ok = "✓" if r['num_labels'] and r['num_labels'] > 0 else "✗"
    cats_ok = "✓" if r['category_list'] and len(r['category_list']) > 0 else "✗"
    print(f"{r['path']:<20} {r['detected_unit']:<6} {r['max_amplitude']:<12.2e} {labels_ok:<8} {cats_ok:<10}")

print(f"基础路径下的所有内容：{list(base_path.iterdir()) if base_path.exists() else '路径不存在'}")

print()
needs_update = [r['path'] for r in results if 'error' not in r and r['detected_unit'] != 'µV']
if needs_update:
    print(f"Datasets needing unit update: {needs_update}")
else:
    print("All datasets are in µV ✓")

[PosixPath('/mnt/dataset2/zlj/TUAE'), PosixPath('/mnt/dataset2/zlj/TUAE4-2'), PosixPath('/mnt/dataset2/zlj/TUAE4-6'), PosixPath('/mnt/dataset2/zlj/code')]
当前遍历目录：/mnt/dataset2/zlj/TUAE
该目录下找到的H5文件：[]
未找到符合条件的H5文件，跳过
当前遍历目录：/mnt/dataset2/zlj/TUAE4-2
该目录下找到的H5文件：[]
未找到符合条件的H5文件，跳过
当前遍历目录：/mnt/dataset2/zlj/TUAE4-6
该目录下找到的H5文件：[PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_1.h5'), PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_2.h5'), PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_3.h5'), PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_4.h5'), PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_5.h5'), PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_6.h5'), PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_7.h5'), PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_8.h5'), PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_9.h5'), PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_10.h5'), PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_11.h5'), PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_12.h5'), PosixPath('/mnt/dataset2/zlj/TUAE4-6/sub_13.h5'), PosixPath('/mnt/dataset2/zlj/TUAE

## Batch Loading Test

In [64]:
loader = load_dataset(data_dir, batch_size=32, num_workers=0)
print(f"Total samples: {len(loader.dataset)}")

for i, batch in enumerate(loader):
    if i >= 3:
        break
    unit, max_amp = detect_unit(batch['data'].numpy())
    print(f"Batch {i}: shape={batch['data'].shape}, unit={unit}, max_amp={max_amp:.2e}, labels={batch['label'].squeeze().tolist()[:5]}...")

Total samples: 1322


RuntimeError: stack expects each tensor to be equal size, but got [24, 500] at entry 0 and [23, 500] at entry 1

## HDF5 Metadata Inspection

In [110]:
h5_files = sorted(Path(data_dir).glob("sub_*.h5"))
if h5_files:
    with HDF5Reader(str(h5_files[0])) as reader:
        subj = reader.subject_attrs
        print(f"Dataset name: {subj.dataset_name}")
        print(f"Task type: {subj.task_type}")
        print(f"Downstream task: {subj.downstream_task_type}")
        print(f"Subject ID: {subj.subject_id}")
        print(f"Sampling rate: {subj.rsFreq} Hz")
        print(f"Montage: {subj.montage}")
        print(f"Channels ({len(subj.chn_name)}): {subj.chn_name[:5]}...")
        print(f"Num labels: {getattr(subj, 'num_labels', 'N/A')}")
        print(f"Categories: {getattr(subj, 'category_list', 'N/A')}")
        print(f"Num trials: {len(reader.get_trial_names())}")
        print(f"Num segments: {len(reader)}")

Dataset name: TUAE_EEG_Dataset
Task type: resting_state
Downstream task: CLASSIFICATION
Subject ID: 1
Sampling rate: 250.0 Hz
Montage: 10_20
Channels (23): ['FP1', 'FP2', 'F3', 'F4', 'C3']...
Num labels: 2
Categories: ['normal', 'abnormal']
Num trials: 1
Num segments: 608
